In [22]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [23]:
spark = (
    SparkSession.builder
    .appName("BronzeLayer")
    .master("local[*]")
    .getOrCreate()
)

print("Spark Version: ",spark.version)

Spark Version:  4.2.0


In [24]:
from pathlib import Path

olist_order_path = Path.cwd().parent/'data'/'raw'/'olist_orders_dataset.csv'

df = spark.read.format('csv')\
.option('inferSchema',False)\
.option('header',True)\
.load(str(olist_order_path))

df.show()


+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  

In [25]:
df = df.withColumn('ingestion_timestamp',current_timestamp())
df.show()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date| ingestion_timestamp|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|2026-10-12 22:08:...|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 1

In [28]:
raw_data_path = Path.cwd().parent/"data"/"bronze"/"orders"

df.write.format('parquet')\
.mode('overwrite')\
.save(str(raw_data_path))



In [30]:
#bronze_df: reading the parquet dataset back into new dataframe

print("number of partition: ",df.rdd.getNumPartitions())

number of partition:  5


In [38]:
#now we ingest multiple files from a folder to another
raw_data_source = Path.cwd().parent/"data"/"raw"
bronze_data_path = Path.cwd().parent/"data"/"bronze"

for file in raw_data_source.glob('*.csv'):
    stemmed_file = file.stem 

    df = (
        spark.read.format('csv')\
        .option("inferSchema",False)\
        .option('header',True)\
        .load(str(file))
    )

    df = (
        df.withColumn('ingestion_timestamp',current_timestamp())
        .withColumn("source_file",lit(file.name))
    )

    output_path = bronze_data_path/stemmed_file
    df.write.format('parquet')\
    .mode('overwrite')\
    .save(str(output_path))

    print(f"Ingested: {file.name}->{output_path}")


Ingested: olist_sellers_dataset.csv->/Users/dipak/Desktop/pyspark-project/data/bronze/olist_sellers_dataset
Ingested: product_category_name_translation.csv->/Users/dipak/Desktop/pyspark-project/data/bronze/product_category_name_translation
Ingested: olist_orders_dataset.csv->/Users/dipak/Desktop/pyspark-project/data/bronze/olist_orders_dataset
Ingested: olist_order_items_dataset.csv->/Users/dipak/Desktop/pyspark-project/data/bronze/olist_order_items_dataset
Ingested: olist_customers_dataset.csv->/Users/dipak/Desktop/pyspark-project/data/bronze/olist_customers_dataset


26/10/12 22:47:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/10/12 22:47:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/10/12 22:47:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/10/12 22:47:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/10/12 22:47:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


Ingested: olist_geolocation_dataset.csv->/Users/dipak/Desktop/pyspark-project/data/bronze/olist_geolocation_dataset
Ingested: olist_order_payments_dataset.csv->/Users/dipak/Desktop/pyspark-project/data/bronze/olist_order_payments_dataset
Ingested: olist_order_reviews_dataset.csv->/Users/dipak/Desktop/pyspark-project/data/bronze/olist_order_reviews_dataset
Ingested: olist_products_dataset.csv->/Users/dipak/Desktop/pyspark-project/data/bronze/olist_products_dataset
